**PS1 — Device Failure Prediction (3-day Binary) — TVM + GATE**

---
SageMaker Studio | MLflow | Feature Store | Model Registry | Model Monitor

Architecture : CUBIC MARS AWS v6
  Gold S3 (Delta) → Feature Store (offline+online) → Train 5 models + stacking
  → MLflow Experiment → MLflow Model Registry → SageMaker Endpoint
  → SageMaker Model Monitor (data quality + model quality)

Target  : will_fail_3d  (0=no failure within 3 days, 1=failure within 3 days)
Scope   : TVM + GATE only (VALIDATOR excluded: zero ServiceNow incidents in incident_root_cause)
Grain   : one row per DEVICE_ID per day
Run in  : SageMaker Studio JupyterLab  (kernel: Python 3 — Data Science 3.0)

---

In [ ]:
# ── CELL 1 : Install dependencies ─────────────────────────────────────────────
import subprocess, sys
pkgs = [
    "s3fs", "pyarrow",
    "scikit-learn>=1.4",
    "xgboost>=2.0", "lightgbm>=4.3", "catboost>=1.2",
    "optuna>=3.6", "shap>=0.45",
    "mlflow>=2.13", "sagemaker-mlflow",
    "boto3", "sagemaker>=2.220,<3.0", "sagemaker-train",
    "pandas>=2.0", "matplotlib", "seaborn",
    "imbalanced-learn",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + pkgs)
print("All packages installed.")

In [ ]:
# ── CELL 2 : Imports ──────────────────────────────────────────────────────────
import os, json, time, warnings
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap, optuna, boto3, sagemaker, joblib, tarfile

# deltalake removed: tables exported as Parquet (DeltaProtocolError on deletionVectors)
from sklearn.model_selection         import TimeSeriesSplit
from sklearn.preprocessing           import StandardScaler, LabelEncoder
from sklearn.pipeline                import Pipeline
from sklearn.linear_model            import LogisticRegression
from sklearn.ensemble                import (RandomForestClassifier,
                                             StackingClassifier)
from sklearn.calibration             import CalibratedClassifierCV
from sklearn.metrics                 import (roc_auc_score, roc_curve,
                                             f1_score, precision_score, recall_score,
                                             average_precision_score, precision_recall_curve,
                                             confusion_matrix, classification_report)
from xgboost   import XGBClassifier
from lightgbm  import LGBMClassifier, early_stopping, log_evaluation
from catboost  import CatBoostClassifier

import mlflow, mlflow.sklearn, mlflow.xgboost
from mlflow.models.signature         import infer_signature

try:
    from sagemaker import Session
except ImportError:
    from sagemaker.session import Session
try:
    from sagemaker import get_execution_role
except ImportError:
    from sagemaker.session import get_execution_role
from sagemaker.feature_store.feature_group      import FeatureGroup
from sagemaker.feature_store.feature_definition import (FeatureDefinition,
                                                        FeatureTypeEnum)
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option("display.max_columns", 60)
print("Imports OK.")

In [ ]:
# ── CELL 3 : Configuration ────────────────────────────────────────────────
# Fill in your artifact bucket name (the dedicated artefacts bucket)
ARTIFACT_BUCKET = "cubic-mars-pm-s3-datalake-dev-artifacts-170202974600"

GOLD_BUCKET = "cubic-mars-pm-s3-datalake-dev-gold-170202974600"
GOLD_PREFIX = "chicago/gold/device_ps1_daily"
GOLD_S3     = f"s3://{GOLD_BUCKET}/{GOLD_PREFIX}"

SM_SESSION  = Session()
ROLE        = get_execution_role()
REGION      = boto3.Session().region_name
SM_CLIENT   = boto3.client("sagemaker", region_name=REGION)
S3_CLIENT   = boto3.client("s3",        region_name=REGION)

# ── MLflow – SageMaker Managed Tracking Server ────────────────────────────
MLFLOW_SERVER_ARN = "arn:aws:sagemaker:us-east-1:170202974600:mlflow-tracking-server/cubic-mars-mlflow-server-dev"
os.environ["MLFLOW_TRACKING_SERVER_ARN"] = MLFLOW_SERVER_ARN

try:
    MLFLOW_TRACKING_URI = SM_CLIENT.describe_mlflow_tracking_server(
        TrackingServerName="cubic-mars-mlflow-server-dev"
    )["TrackingServerUrl"]
    print(f"MLflow managed server: {MLFLOW_TRACKING_URI}")
except Exception:
    MLFLOW_TRACKING_URI = "file:///home/sagemaker-user/mlruns"
    print("MLflow managed server not found – using local file fallback (EFS)")

mlflow.set_tracking_uri(MLFLOW_SERVER_ARN)

# Per-category names (experiment / model / registry / endpoint) are built
# inside CELL 6b and overridden at the top of each CATEGORY loop.
GOLD_VERSION = 6

# ── Feature Store ─────────────────────────────────────────────────────────
# Online store = latest record per device for real-time inference.
# IAM role now has AmazonSageMakerFeatureStoreAccess on the artifacts bucket.
ENABLE_FEATURE_STORE = True
FS_GROUP_NAME        = "ps1-chicago-device-features-dev"

# ── Target horizon ────────────────────────────────────────────────────────
# "3d" → predict failure within 3 days (tighter signal, higher precision)
# "7d" → predict failure within 7 days (more positives, easier AUC signal)
TARGET_HORIZON = "3d"

print(f"Region               : {REGION}")
print(f"Gold S3              : {GOLD_S3}")
print(f"Artifacts            : s3://{ARTIFACT_BUCKET}/")
print(f"MLflow URI           : {MLFLOW_TRACKING_URI}")
print(f"Enable Feature Store : {ENABLE_FEATURE_STORE}  ({FS_GROUP_NAME})")
print(f"Target horizon       : will_fail_{TARGET_HORIZON}")


In [ ]:
# ── CELL 4 : Load Gold table (Parquet on S3) ──────────────────────────────────
# G01 source: mars_dev.gold.device_ps1_daily exported via notebooks/export_gold_to_s3.py
# Column note: gold SQL writes "transit_day"; aliased to "event_date" below.

print(f"Loading Gold table from: {GOLD_S3}")

df = pd.read_parquet(GOLD_S3, storage_options={"client_kwargs": {"region_name": REGION}})
GOLD_VERSION = None   # Parquet has no version; use None for MLflow tags
DATA_SOURCE  = "parquet"

# gold SQL writes transit_day; alias to event_date used throughout notebook
if "transit_day" in df.columns and "event_date" not in df.columns:
    df = df.rename(columns={"transit_day": "event_date"})
df["event_date"] = pd.to_datetime(df["event_date"])

print(f"Parquet source  : confirmed")
print(f"Shape           : {df.shape}")
print(f"Date range      : {df['event_date'].min().date()}  →  {df['event_date'].max().date()}")
print(f"\nTarget distribution:\n{df['will_fail_3d'].value_counts()}")
print(f"Failure rate    : {df['will_fail_3d'].mean():.2%}")
print(f"\nDevice mix:\n{df.drop_duplicates('DEVICE_ID')['mars_device_category'].value_counts()}")

In [ ]:
# ── CELL 5 : EDA ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

df["will_fail_3d"].value_counts().plot(
    kind="bar", ax=axes[0], color=["#43A047","#E53935"], edgecolor="white")
axes[0].set_title("Target: will_fail_3d")

(1 - df.isnull().mean()).sort_values().head(20).plot(
    kind="barh", ax=axes[1], color="#1565C0")
axes[1].set_title("Feature completeness (lowest 20)")

df.groupby("event_date")["will_fail_3d"].sum().plot(
    ax=axes[2], color="#E53935", linewidth=0.8)
axes[2].set_title("Daily failure count over time")

plt.tight_layout()
plt.savefig("/tmp/ps1_eda.png", dpi=100)
plt.show()
print("EDA saved → /tmp/ps1_eda.png")

In [ ]:
# ── CELL 6 : Feature engineering (global — all categories) ──────────────────
TARGET   = f"will_fail_{TARGET_HORIZON}"

# R3-2: Exclude VALIDATOR
df = df[df["mars_device_category"].isin(["TVM", "GATE"])].copy()
print(f"After VALIDATOR exclusion: {df.shape}  ({df['mars_device_category'].value_counts().to_dict()})")

# Sort once — all time-windowed features below depend on this
df = df.sort_values(["DEVICE_ID", "event_date"]).reset_index(drop=True)

ID_COLS  = ["DEVICE_ID", "event_date", "split"]

# ── Cast decimal.Decimal columns to float64 ───────────────────────────────────
# Delta Lake / Databricks exports DECIMAL columns as Python decimal.Decimal objects.
# pandas cannot do arithmetic between decimal.Decimal and float — cast everything up front.
import decimal as _decimal
_decimal_cols = [
    col for col in df.columns
    if df[col].dtype == object
    and len(df[col].dropna()) > 0
    and isinstance(df[col].dropna().iloc[0], _decimal.Decimal)
]
if _decimal_cols:
    df[_decimal_cols] = df[_decimal_cols].apply(pd.to_numeric, errors='coerce')
    print(f"Cast {len(_decimal_cols)} decimal.Decimal column(s) to float64: {_decimal_cols}")

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK A — Rolling aggregates (original)
# ══════════════════════════════════════════════════════════════════════════════
for col, window in [("oos_flag", 7), ("oos_flag", 30), ("daily_event_count", 7)]:
    if col in df.columns:
        df[f"{col}_roll{window}"] = (
            df.groupby("DEVICE_ID")[col]
              .transform(lambda x: x.rolling(window, min_periods=1).sum()))

if "daily_event_count" in df.columns:
    df["event_rate_zscore"] = (
        df.groupby("DEVICE_ID")["daily_event_count"]
          .transform(lambda x: (x - x.mean()) / (x.std() + 1e-6)))

if "use_revenue_decline_flag" in df.columns:
    df["use_revenue_decline_flag"] = df["use_revenue_decline_flag"].astype(int)

if "use_revenue_cents" in df.columns:
    df["use_revenue_cents_roll7"] = (
        df.groupby("DEVICE_ID")["use_revenue_cents"]
          .transform(lambda x: x.rolling(7, min_periods=1).mean()))

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK B — Lag features t-1 / t-2 / t-3  (shift within device, no leakage)
# ══════════════════════════════════════════════════════════════════════════════
LAG_COLS = [c for c in ["oos_flag", "daily_event_count", "use_revenue_cents"] if c in df.columns]
for col in LAG_COLS:
    for lag in [1, 2, 3]:
        df[f"{col}_lag{lag}"] = df.groupby("DEVICE_ID")[col].shift(lag)

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK C — Historical OOS counts + days-since + lifetime  (all shifted)
# ══════════════════════════════════════════════════════════════════════════════
if "oos_flag" in df.columns:
    for window in [30, 90]:
        df[f"oos_count_{window}d"] = (
            df.groupby("DEVICE_ID")["oos_flag"]
              .transform(lambda x, w=window: x.shift(1).rolling(w, min_periods=1).sum()))

    def _days_since_last_oos(x):
        cs = x.shift(1).cumsum()
        result = cs.groupby(cs).cumcount().astype(float)
        result[cs == 0] = np.nan
        return result

    df["days_since_last_oos"] = (
        df.groupby("DEVICE_ID")["oos_flag"].transform(_days_since_last_oos))

    df["oos_lifetime_count"] = (
        df.groupby("DEVICE_ID")["oos_flag"]
          .transform(lambda x: x.shift(1).cumsum()))

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK D — Device age
# ══════════════════════════════════════════════════════════════════════════════
df["device_age_days"] = df.groupby("DEVICE_ID")["event_date"].transform(
    lambda x: (x - x.min()).dt.days)

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK E — Consecutive OOS streak  (acute deterioration signal)
# ══════════════════════════════════════════════════════════════════════════════
if "oos_flag" in df.columns:
    def _consec_oos(x):
        s = x.shift(1).fillna(0)
        grp = (s != s.shift(1)).cumsum()
        streak = s.groupby(grp).cumcount() + 1
        streak[s == 0] = 0.0
        return streak.astype(float)

    df["consec_oos_days"] = df.groupby("DEVICE_ID")["oos_flag"].transform(_consec_oos)
    df["max_streak_30d"]  = (
        df.groupby("DEVICE_ID")["consec_oos_days"]
          .transform(lambda x: x.shift(1).rolling(30, min_periods=1).max()))

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK F — OOS rate acceleration  (trend: worsening or recovering?)
# ══════════════════════════════════════════════════════════════════════════════
if "oos_flag" in df.columns:
    _oos_7d_recent = df.groupby("DEVICE_ID")["oos_flag"].transform(
        lambda x: x.shift(1).rolling(7, min_periods=1).sum())
    _oos_7d_prev   = df.groupby("DEVICE_ID")["oos_flag"].transform(
        lambda x: x.shift(8).rolling(7, min_periods=1).sum())
    df["oos_rate_accel_7d"] = (_oos_7d_recent - _oos_7d_prev).fillna(0)

    _oos_30d_recent = df.groupby("DEVICE_ID")["oos_flag"].transform(
        lambda x: x.shift(1).rolling(30, min_periods=1).sum())
    _oos_30d_prev   = df.groupby("DEVICE_ID")["oos_flag"].transform(
        lambda x: x.shift(31).rolling(30, min_periods=1).sum())
    df["oos_rate_accel_30d"] = (_oos_30d_recent - _oos_30d_prev).fillna(0)

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK G — Exponentially weighted OOS  (recent OOS weighted higher than old)
# ══════════════════════════════════════════════════════════════════════════════
if "oos_flag" in df.columns:
    df["oos_ewm7"]  = (
        df.groupby("DEVICE_ID")["oos_flag"]
          .transform(lambda x: x.shift(1).ewm(span=7,  min_periods=1).mean()))
    df["oos_ewm30"] = (
        df.groupby("DEVICE_ID")["oos_flag"]
          .transform(lambda x: x.shift(1).ewm(span=30, min_periods=1).mean()))

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK H — Recovery count  (OOS→OK transitions = recurrence-prone devices)
# ══════════════════════════════════════════════════════════════════════════════
if "oos_flag" in df.columns:
    _oos_lag1 = df.groupby("DEVICE_ID")["oos_flag"].shift(1).fillna(0)
    _oos_lag2 = df.groupby("DEVICE_ID")["oos_flag"].shift(2).fillna(0)
    df["_recovery"] = ((_oos_lag2 == 1) & (_oos_lag1 == 0)).astype(float)
    df["recovery_count_30d"] = (
        df.groupby("DEVICE_ID")["_recovery"]
          .transform(lambda x: x.rolling(30, min_periods=1).sum()))
    df["recovery_count_90d"] = (
        df.groupby("DEVICE_ID")["_recovery"]
          .transform(lambda x: x.rolling(90, min_periods=1).sum()))
    df = df.drop(columns=["_recovery"])

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK I — Anomaly z-scores  (deviation from device's own rolling baseline)
# ══════════════════════════════════════════════════════════════════════════════
if "daily_event_count" in df.columns:
    _mu  = df.groupby("DEVICE_ID")["daily_event_count"].transform(
               lambda x: x.shift(1).rolling(30, min_periods=5).mean())
    _std = df.groupby("DEVICE_ID")["daily_event_count"].transform(
               lambda x: x.shift(1).rolling(30, min_periods=5).std())
    df["event_count_zscore_30d"] = ((df["daily_event_count"] - _mu) / (_std + 1e-6)).fillna(0)

if "use_revenue_cents" in df.columns:
    _mu  = df.groupby("DEVICE_ID")["use_revenue_cents"].transform(
               lambda x: x.shift(1).rolling(30, min_periods=5).mean())
    _std = df.groupby("DEVICE_ID")["use_revenue_cents"].transform(
               lambda x: x.shift(1).rolling(30, min_periods=5).std())
    df["revenue_zscore_30d"] = ((df["use_revenue_cents"].astype(float) - _mu) / (_std + 1e-6)).fillna(0)

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK J — Calendar features
# ══════════════════════════════════════════════════════════════════════════════
df["day_of_week"]  = df["event_date"].dt.dayofweek
df["month"]        = df["event_date"].dt.month
df["day_of_month"] = df["event_date"].dt.day
df["is_weekend"]   = (df["day_of_week"] >= 5).astype(int)
df["quarter"]      = df["event_date"].dt.quarter

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK K — Peer comparison  (facility/location-level failure rate)
# ══════════════════════════════════════════════════════════════════════════════
_facility_col = next((c for c in df.columns
                      if any(k in c.lower() for k in ["facility", "station", "location", "site"])), None)
if _facility_col and "oos_flag" in df.columns:
    _peer_oos = (df.groupby(["event_date", _facility_col])["oos_flag"]
                   .transform("mean"))
    df["peer_oos_rate"] = _peer_oos
    df["oos_vs_peer"]   = df["oos_flag"] - _peer_oos

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK L — Interaction features
# ══════════════════════════════════════════════════════════════════════════════
if "oos_flag" in df.columns and "daily_event_count" in df.columns:
    df["oos_x_event_count"] = (
        df.groupby("DEVICE_ID")["oos_flag"].shift(1).fillna(0)
        * df.groupby("DEVICE_ID")["daily_event_count"].shift(1).fillna(0))

if "oos_ewm7" in df.columns and "device_age_days" in df.columns:
    df["oos_ewm7_x_age"] = df["oos_ewm7"] * np.log1p(df["device_age_days"])

# ══════════════════════════════════════════════════════════════════════════════
# LABEL LEAK EXCLUSION + FEATURE_COLS
# ══════════════════════════════════════════════════════════════════════════════
LABEL_LEAK_COLS = [
    "failure_level",
    "is_chargeable",
    "is_hardware_oos",
    "is_hardware_event",
    "duration_min",
    "oos_duration_min",
    "will_fail_7d",
    "will_fail_3d",   # exclude both horizons — active TARGET excluded separately
]

FEATURE_COLS = [
    c for c in df.columns
    if c not in ID_COLS + [TARGET] + LABEL_LEAK_COLS
    and pd.api.types.is_numeric_dtype(df[c])
]

print(f"\n{'─'*60}")
print(f"Total features : {len(FEATURE_COLS)}")
blocks = {
    "A  rolling aggregates"  : [c for c in FEATURE_COLS if "_roll" in c or "zscore" == c.split("_")[-1]],
    "B  lag t-1/2/3"         : [c for c in FEATURE_COLS if "_lag" in c],
    "C  OOS history"         : [c for c in FEATURE_COLS if "oos_count" in c or "days_since" in c or "lifetime" in c],
    "D  device age"          : [c for c in FEATURE_COLS if "device_age" in c],
    "E  streak"              : [c for c in FEATURE_COLS if "streak" in c or "consec" in c],
    "F  acceleration"        : [c for c in FEATURE_COLS if "accel" in c],
    "G  EWM"                 : [c for c in FEATURE_COLS if "ewm" in c and "x_" not in c],
    "H  recovery"            : [c for c in FEATURE_COLS if "recovery" in c],
    "I  anomaly z-scores"    : [c for c in FEATURE_COLS if "zscore" in c],
    "J  calendar"            : [c for c in FEATURE_COLS if c in ["day_of_week","month","day_of_month","is_weekend","quarter"]],
    "K  peer"                : [c for c in FEATURE_COLS if "peer" in c or "vs_peer" in c],
    "L  interactions"        : [c for c in FEATURE_COLS if "_x_" in c],
}
for name, cols in blocks.items():
    if cols:
        print(f"  Block {name:<28}: {len(cols):2d} features")
print(f"{'─'*60}")
print(f"Leak cols excluded: {[c for c in LABEL_LEAK_COLS if c in df.columns]}")
print("Feature engineering complete — run CELL 6b → CELL 10 → CELLs 11-23")

In [ ]:
# ── CELL 6b : Per-category temporal splits & resource config ─────────────────
# TVM and GATE trained as SEPARATE models.
# Each gets its own: split, scale_pos_weight, MLflow experiment,
# Feature Store group, Model Registry, and SageMaker endpoint.

CATEGORIES = {}
for cat in ["TVM", "GATE"]:
    cat_df = df[df["mars_device_category"] == cat].sort_values(["DEVICE_ID", "event_date"])

    tr = cat_df[cat_df["event_date"] <  "2025-10-01"]
    va = cat_df[(cat_df["event_date"] >= "2025-10-01") & (cat_df["event_date"] < "2026-01-01")]
    te = cat_df[cat_df["event_date"] >= "2026-01-01"]

    n0, n1 = (tr[TARGET] == 0).sum(), (tr[TARGET] == 1).sum()

    CATEGORIES[cat] = {
        "train_df": tr, "val_df": va, "test_df": te,
        "X_train": tr[FEATURE_COLS].values, "y_train": tr[TARGET].values,
        "X_val":   va[FEATURE_COLS].values, "y_val":   va[TARGET].values,
        "X_test":  te[FEATURE_COLS].values, "y_test":  te[TARGET].values,
        "X_cv":    np.vstack([tr[FEATURE_COLS].values, va[FEATURE_COLS].values]),
        "y_cv":    np.concatenate([tr[TARGET].values, va[TARGET].values]),
        "scale_pos_weight": round(n0 / max(n1, 1), 2),
        "neg": n0, "pos": n1,
        "experiment_name": f"chicago-ps1-3d-{cat.lower()}-failure",
        "model_name":      f"ps1-3d-{cat.lower()}-device-failure",
        "registry_name":   f"chicago-ps1-3d-{cat.lower()}-failure",
        "fs_group":        f"chicago-ps1-3d-{cat.lower()}-features",
        "endpoint":        f"chicago-ps1-3d-{cat.lower()}-failure-v1",
        # Recall floor for threshold tuning on val PR curve.
        # GATE: only 0.60% positive rate (scale_pos_weight ~166) —
        # recall=0.80 may not be achievable; use 0.70 as the floor.
        "recall_floor": 0.80 if cat == "TVM" else 0.70,
    }
    mlflow.set_experiment(CATEGORIES[cat]["experiment_name"])

    print(f"\n── {cat} ──────────────────────────────────────────────────────")
    print(f"  Train  : {len(tr):>8,}  ({tr['event_date'].min().date()} -> {tr['event_date'].max().date()})")
    print(f"  Val    : {len(va):>8,}")
    print(f"  Test   : {len(te):>8,}")
    print(f"  Pos rate (train): {n1/max(len(tr),1):.2%}  ({n1:,} positives)")
    print(f"  scale_pos_weight: {CATEGORIES[cat]['scale_pos_weight']}")
    print(f"  MLflow experiment: {CATEGORIES[cat]['experiment_name']}")


In [ ]:
# ── CELL 7 : Feature Store — create group + seed ONLINE store ───────────────
# One combined Feature Group; online store = latest record/device for inference.
# Note: df['split'] is added in CELL 9 — not available here. Use full df,
# latest-per-device gives the most recent device state for online inference.
if not ENABLE_FEATURE_STORE:
    print("Feature Store disabled (ENABLE_FEATURE_STORE=False) — skipping CELL 7.")
    print("Set ENABLE_FEATURE_STORE=True in CELL 3 once the role has AmazonSageMakerFeatureStoreAccess.")
else:
    from sagemaker.feature_store.feature_group import FeatureGroup
    from sagemaker.feature_store.feature_definition import FeatureDefinition, FeatureTypeEnum
    import time

    OFFLINE_URI = f"s3://{ARTIFACT_BUCKET}/feature-store/{FS_GROUP_NAME}"

    # Latest record per device across all rows (split col not yet available)
    fs_cols   = [c for c in FEATURE_COLS + [TARGET, "DEVICE_ID", "event_date"] if c in df.columns]
    fs_full   = df[fs_cols].copy()
    fs_full["DEVICE_ID"] = fs_full["DEVICE_ID"].astype(str)
    fs_latest = (fs_full.sort_values("event_date")
                         .groupby("DEVICE_ID", as_index=False).tail(1)
                         .reset_index(drop=True))
    fs_latest["event_time"] = fs_latest["event_date"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    fs_latest = fs_latest.drop(columns=["event_date"])
    print(f"Online-store records (latest/device): {len(fs_latest):,}  (from {len(fs_full):,} total rows)")

    def fs_type(dt):
        if pd.api.types.is_float_dtype(dt):   return FeatureTypeEnum.FRACTIONAL
        if pd.api.types.is_integer_dtype(dt): return FeatureTypeEnum.INTEGRAL
        return FeatureTypeEnum.STRING

    feature_group = FeatureGroup(
        name=FS_GROUP_NAME, sagemaker_session=SM_SESSION,
        feature_definitions=[FeatureDefinition(c, fs_type(fs_latest[c].dtype))
                             for c in fs_latest.columns])

    # Idempotent create — self-heals a prior CreateFailed
    def fg_status():
        try: return feature_group.describe().get("FeatureGroupStatus")
        except Exception: return None

    st = fg_status()
    if st == "CreateFailed":
        print("Prior CreateFailed — deleting & recreating")
        feature_group.delete(); time.sleep(20); st = None
    if st is None:
        feature_group.create(
            s3_uri=OFFLINE_URI, record_identifier_name="DEVICE_ID",
            event_time_feature_name="event_time", role_arn=ROLE,
            enable_online_store=True,
            description="PS1 daily device features — Chicago Ventra (online = latest/device)")
        while fg_status() != "Created":
            time.sleep(15); print("  status:", fg_status())
    print(f"Feature Group ready: {FS_GROUP_NAME}  (status: {fg_status()})")

    # Seed online store + verify
    feature_group.ingest(data_frame=fs_latest.reset_index(drop=True), max_workers=4, wait=True)
    sample_id = fs_latest["DEVICE_ID"].iloc[0]
    rec = feature_group.get_record(record_identifier_value_as_string=sample_id)
    print(f"Ingested {len(fs_latest):,} records | online sample ({sample_id}):",
          {f['FeatureName']: f['ValueAsString'] for f in (rec or [])[:4]})
    globals()['feature_group'] = feature_group


In [ ]:
# ── CELL 8 : Feature Store — spot-check online store per category ────────────
# Creation and ingestion are in CELL 7. Verifies one TVM + one GATE device.
if not ENABLE_FEATURE_STORE:
    print("Feature Store disabled — skipping CELL 8.")
else:
    for cat_name in ["TVM", "GATE"]:
        sample_ids = (df[df["mars_device_category"] == cat_name]["DEVICE_ID"]
                      .astype(str).unique()[:1])
        if len(sample_ids):
            rec = feature_group.get_record(record_identifier_value_as_string=sample_ids[0])
            print(f"[{cat_name}] online store ({sample_ids[0]}):",
                  {f['FeatureName']: f['ValueAsString'] for f in (rec or [])[:4]})
        else:
            print(f"[{cat_name}] no devices found for spot-check")
    print("Feature Store verification complete.")


In [ ]:
# ── CELL 9 : Helper functions (category-aware via loop globals) ──────────────
# X_train/val/test, y_train/val/test, SCALE_POS_WEIGHT, COMMON_TAGS
# are reassigned at the start of each CATEGORY iteration.

def make_common_tags(category):
    return {
        "ps"           : "PS1_3d",
        "category"     : category,
        "target"       : "will_fail_3d",
        "city"         : "chicago",
        "gold_version" : str(GOLD_VERSION),
        "grain"        : "device_day",
        "architecture" : "CUBIC-MARS-AWS-v6",
    }

def evaluate(model, X, y, split="test"):
    prob = model.predict_proba(X)[:, 1]
    pred = (prob >= 0.5).astype(int)
    return {
        f"{split}_auc"  : round(roc_auc_score(y, prob), 4),
        f"{split}_ap"   : round(average_precision_score(y, prob), 4),
        f"{split}_f1"   : round(f1_score(y, pred), 4),
        f"{split}_prec" : round(precision_score(y, pred, zero_division=0), 4),
        f"{split}_rec"  : round(recall_score(y, pred, zero_division=0), 4),
    }

def log_cm(model, X, y, name, split="test"):
    pred = (model.predict_proba(X)[:, 1] >= 0.5).astype(int)
    cm   = confusion_matrix(y, pred)
    fig, ax = plt.subplots(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["No Fail","Fail"], yticklabels=["No Fail","Fail"])
    ax.set_title(f"{name} — Confusion Matrix ({split})")
    path = f"/tmp/cm_{name}_{split}.png"
    fig.savefig(path, bbox_inches="tight", dpi=80); plt.show(); plt.close(fig)
    mlflow.log_artifact(path)

def plot_roc_pr(model, name):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    colors = {"train": "#1565C0", "val": "#E65100", "test": "#2E7D32"}
    for label, X, y in [("train", X_train, y_train), ("val", X_val, y_val), ("test", X_test, y_test)]:
        prob        = model.predict_proba(X)[:, 1]
        fpr, tpr, _ = roc_curve(y, prob)
        pr, rc, _   = precision_recall_curve(y, prob)
        col         = colors[label]
        axes[0].plot(fpr, tpr, color=col, lw=1.6, label=f"{label}  AUC={roc_auc_score(y,prob):.4f}")
        axes[1].plot(rc,  pr,  color=col, lw=1.6, label=f"{label}  AP={average_precision_score(y,prob):.4f}")
    axes[0].plot([0,1],[0,1],"k--", lw=0.7)
    axes[0].set(xlabel="FPR", ylabel="TPR", title=f"{name} — ROC"); axes[0].legend(fontsize=9)
    axes[1].set(xlabel="Recall", ylabel="Precision", title=f"{name} — PR"); axes[1].legend(fontsize=9)
    fig.suptitle(f"{name}  |  Train/Val/Test", fontsize=11, y=1.01); fig.tight_layout()
    path = f"/tmp/roc_pr_{name}.png"
    fig.savefig(path, dpi=100, bbox_inches="tight"); plt.show(); plt.close(fig)
    mlflow.log_artifact(path)

def log_feat_importance(importances, name, top_n=20):
    fi = pd.Series(importances, index=FEATURE_COLS).nlargest(top_n)
    fig, ax = plt.subplots(figsize=(9, 5))
    fi.sort_values().plot(kind="barh", ax=ax, color="#1565C0", edgecolor="white")
    ax.set_title(f"{name} — Top {top_n} Feature Importances"); ax.set_xlabel("Importance")
    fig.tight_layout()
    path = f"/tmp/fi_{name}.png"
    fig.savefig(path, dpi=100); plt.show(); plt.close(fig); mlflow.log_artifact(path)
    print(f"\n-- {name} -- Top 10"); print(fi.head(10).to_string())

def log_clf_report(model, name):
    for split, X, y in [("train", X_train, y_train), ("val", X_val, y_val), ("test", X_test, y_test)]:
        pred = (model.predict_proba(X)[:, 1] >= 0.5).astype(int)
        print(f"\n-- {name} [{split}]")
        print(classification_report(y, pred, target_names=["No Fail", "Fail"], digits=4))
    pred_test = (model.predict_proba(X_test)[:, 1] >= 0.5).astype(int)
    rep = classification_report(y_test, pred_test, target_names=["No Fail","Fail"], digits=4)
    path = f"/tmp/report_{name}.txt"
    with open(path, "w") as fh: fh.write(rep)
    mlflow.log_artifact(path)

print("Helpers defined.")


In [ ]:
# ── CELL 10 : Per-category setup — imputation + SMOTE + CATEGORY_RESULTS ──────
# Run once before any training cell. CATEGORY_RESULTS accumulates across cells below.

CATEGORY_RESULTS = {cat: {} for cat in ['TVM', 'GATE']}

for CATEGORY in ['TVM', 'GATE']:
    cat = CATEGORIES[CATEGORY]

    # Cast to float64 before imputation (integer/object dtypes break nanmedian)
    for key in ('X_train', 'X_val', 'X_test', 'X_cv'):
        cat[key] = cat[key].astype(float)

    # Train-only median imputation — no leakage from val/test
    train_median = np.nanmedian(cat['X_train'], axis=0)
    cat['X_train'] = np.where(np.isnan(cat['X_train']), train_median, cat['X_train'])
    cat['X_val']   = np.where(np.isnan(cat['X_val']),   train_median, cat['X_val'])
    cat['X_test']  = np.where(np.isnan(cat['X_test']),  train_median, cat['X_test'])
    cat['X_cv']    = np.where(np.isnan(cat['X_cv']),    train_median, cat['X_cv'])
    CATEGORY_RESULTS[CATEGORY]['train_median'] = train_median

    # SMOTE oversampling for GATE only (0.6% positive → ~5% after resampling)
    # Applied to X_train/y_train only — val/test stay original for unbiased eval.
    if CATEGORY == 'GATE':
        from imblearn.over_sampling import SMOTE
        _sm = SMOTE(sampling_strategy=0.05, random_state=42)
        cat['X_train'], cat['y_train'] = _sm.fit_resample(cat['X_train'], cat['y_train'])
        cat['pos'] = int(cat['y_train'].sum())
        _n_smoted  = len(cat['y_train'])
        cat['scale_pos_weight'] = round((_n_smoted - cat['pos']) / cat['pos'], 2)

    n_total = len(cat['X_train'])
    print(f'\n── {CATEGORY} ──────────────────────────────────────────')
    print(f"  Train  : {n_total:,} rows  ({cat['pos']:,} pos = {cat['pos']/n_total:.2%})"
          + (" [SMOTE applied]" if CATEGORY == 'GATE' else ""))
    print(f"  Val    : {len(cat['X_val']):,}")
    print(f"  Test   : {len(cat['X_test']):,}")
    print(f"  scale_pos_weight : {cat['scale_pos_weight']}")
    print(f"  recall_floor     : {cat['recall_floor']}")
    print(f"  Experiment       : {cat['experiment_name']}")

print('\nSetup complete — run CELLs 11-23 in order.')


In [ ]:
# ── CELL 11 : MODEL 1 — Logistic Regression (per category) ─────────────────────
for CATEGORY in ["TVM", "GATE"]:
    cat = CATEGORIES[CATEGORY]
    X_train = cat["X_train"]; y_train = cat["y_train"]
    X_val   = cat["X_val"];   y_val   = cat["y_val"]
    X_test  = cat["X_test"];  y_test  = cat["y_test"]
    X_cv    = cat["X_cv"];    y_cv    = cat["y_cv"]
    train_df = cat["train_df"]; val_df = cat["val_df"]; test_df = cat["test_df"]
    SCALE_POS_WEIGHT  = cat["scale_pos_weight"]
    EXPERIMENT_NAME   = cat["experiment_name"]
    MLFLOW_MODEL_NAME = cat["model_name"]
    MODEL_REGISTRY_NAME = cat["registry_name"]
    RECALL_FLOOR      = cat["recall_floor"]
    COMMON_TAGS       = {**make_common_tags(CATEGORY),
                         "run_date": pd.Timestamp.today().strftime("%Y-%m-%d")}
    mlflow.set_experiment(EXPERIMENT_NAME)
    tag = CATEGORY.lower()

    with mlflow.start_run(run_name=f'lr_{tag}') as run:
        mlflow.set_tags({**COMMON_TAGS, 'model_type': 'linear_baseline'})
        lr = Pipeline([('sc', StandardScaler()),
                        ('clf', LogisticRegression(C=0.1, class_weight='balanced',
                                                   max_iter=1000, solver='lbfgs', random_state=42))])
        lr.fit(X_train, y_train)
        vm = evaluate(lr, X_val, y_val, 'val'); tm = evaluate(lr, X_test, y_test, 'test')
        mlflow.log_params({'C': 0.1, 'class_weight': 'balanced', 'scaler': 'standard'})
        mlflow.log_metrics({**vm, **tm})
        log_cm(lr, X_test, y_test, f'lr_{tag}')
        plot_roc_pr(lr, f'lr_{tag}')
        log_feat_importance(np.abs(lr.named_steps['clf'].coef_[0]), f'lr_{tag}')
        log_clf_report(lr, f'lr_{tag}')
        mlflow.sklearn.log_model(lr, 'model',
                                 signature=infer_signature(X_train, lr.predict_proba(X_train)),
                                 input_example=X_train[:3])
        CATEGORY_RESULTS[CATEGORY]['lr_run'] = run.info.run_id
    print(f'  [{CATEGORY}] LR   AUC={tm["test_auc"]}  AP={tm["test_ap"]}  F1={tm["test_f1"]}')


In [ ]:
# ── CELL 12 : MODEL 2 — Random Forest (per category) ───────────────────────────
for CATEGORY in ["TVM", "GATE"]:
    cat = CATEGORIES[CATEGORY]
    X_train = cat["X_train"]; y_train = cat["y_train"]
    X_val   = cat["X_val"];   y_val   = cat["y_val"]
    X_test  = cat["X_test"];  y_test  = cat["y_test"]
    X_cv    = cat["X_cv"];    y_cv    = cat["y_cv"]
    train_df = cat["train_df"]; val_df = cat["val_df"]; test_df = cat["test_df"]
    SCALE_POS_WEIGHT  = cat["scale_pos_weight"]
    EXPERIMENT_NAME   = cat["experiment_name"]
    MLFLOW_MODEL_NAME = cat["model_name"]
    MODEL_REGISTRY_NAME = cat["registry_name"]
    RECALL_FLOOR      = cat["recall_floor"]
    COMMON_TAGS       = {**make_common_tags(CATEGORY),
                         "run_date": pd.Timestamp.today().strftime("%Y-%m-%d")}
    mlflow.set_experiment(EXPERIMENT_NAME)
    tag = CATEGORY.lower()

    with mlflow.start_run(run_name=f'rf_{tag}') as run:
        mlflow.set_tags({**COMMON_TAGS, 'model_type': 'bagging_ensemble'})
        rf = RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=20,
                                     class_weight='balanced_subsample', n_jobs=-1, random_state=42)
        rf.fit(X_train, y_train)
        vm = evaluate(rf, X_val, y_val, 'val'); tm = evaluate(rf, X_test, y_test, 'test')
        mlflow.log_params({'n_estimators': 300, 'max_depth': 12, 'class_weight': 'balanced_subsample'})
        mlflow.log_metrics({**vm, **tm})
        log_cm(rf, X_test, y_test, f'rf_{tag}')
        plot_roc_pr(rf, f'rf_{tag}')
        log_feat_importance(rf.feature_importances_, f'rf_{tag}')
        log_clf_report(rf, f'rf_{tag}')
        mlflow.sklearn.log_model(rf, 'model',
                                 signature=infer_signature(X_train, rf.predict_proba(X_train)),
                                 input_example=X_train[:3])
        CATEGORY_RESULTS[CATEGORY]['rf_run'] = run.info.run_id
    print(f'  [{CATEGORY}] RF   AUC={tm["test_auc"]}  AP={tm["test_ap"]}  F1={tm["test_f1"]}')


In [ ]:
# ── CELL 13 : MODEL 3 — XGBoost baseline (per category) ────────────────────────
for CATEGORY in ["TVM", "GATE"]:
    cat = CATEGORIES[CATEGORY]
    X_train = cat["X_train"]; y_train = cat["y_train"]
    X_val   = cat["X_val"];   y_val   = cat["y_val"]
    X_test  = cat["X_test"];  y_test  = cat["y_test"]
    X_cv    = cat["X_cv"];    y_cv    = cat["y_cv"]
    train_df = cat["train_df"]; val_df = cat["val_df"]; test_df = cat["test_df"]
    SCALE_POS_WEIGHT  = cat["scale_pos_weight"]
    EXPERIMENT_NAME   = cat["experiment_name"]
    MLFLOW_MODEL_NAME = cat["model_name"]
    MODEL_REGISTRY_NAME = cat["registry_name"]
    RECALL_FLOOR      = cat["recall_floor"]
    COMMON_TAGS       = {**make_common_tags(CATEGORY),
                         "run_date": pd.Timestamp.today().strftime("%Y-%m-%d")}
    mlflow.set_experiment(EXPERIMENT_NAME)
    tag = CATEGORY.lower()

    XGB_PARAMS = dict(n_estimators=600, max_depth=6, learning_rate=0.05, subsample=0.8,
                      colsample_bytree=0.8, min_child_weight=10, gamma=1.0, reg_alpha=0.1,
                      reg_lambda=1.0, scale_pos_weight=SCALE_POS_WEIGHT, eval_metric='auc',
                      early_stopping_rounds=40, tree_method='hist', random_state=42, n_jobs=-1)
    with mlflow.start_run(run_name=f'xgb_{tag}') as run:
        mlflow.set_tags({**COMMON_TAGS, 'model_type': 'gradient_boosting_xgb'})
        xgb = XGBClassifier(**XGB_PARAMS)
        xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)
        vm = evaluate(xgb, X_val, y_val, 'val'); tm = evaluate(xgb, X_test, y_test, 'test')
        mlflow.log_params({**XGB_PARAMS, 'best_iter': xgb.best_iteration})
        mlflow.log_metrics({**vm, **tm})
        log_cm(xgb, X_test, y_test, f'xgb_{tag}')
        plot_roc_pr(xgb, f'xgb_{tag}')
        log_feat_importance(xgb.feature_importances_, f'xgb_{tag}')
        log_clf_report(xgb, f'xgb_{tag}')
        sig = infer_signature(pd.DataFrame(X_train, columns=FEATURE_COLS), xgb.predict_proba(X_train))
        mlflow.xgboost.log_model(xgb, 'model', signature=sig,
                                  input_example=pd.DataFrame(X_train[:3], columns=FEATURE_COLS))
        CATEGORY_RESULTS[CATEGORY]['xgb_run'] = run.info.run_id
    print(f'  [{CATEGORY}] XGB  AUC={tm["test_auc"]}  AP={tm["test_ap"]}  F1={tm["test_f1"]}  best_iter={xgb.best_iteration}')


In [ ]:
# ── CELL 14 : MODEL 4 — LightGBM (per category) ────────────────────────────────
for CATEGORY in ["TVM", "GATE"]:
    cat = CATEGORIES[CATEGORY]
    X_train = cat["X_train"]; y_train = cat["y_train"]
    X_val   = cat["X_val"];   y_val   = cat["y_val"]
    X_test  = cat["X_test"];  y_test  = cat["y_test"]
    X_cv    = cat["X_cv"];    y_cv    = cat["y_cv"]
    train_df = cat["train_df"]; val_df = cat["val_df"]; test_df = cat["test_df"]
    SCALE_POS_WEIGHT  = cat["scale_pos_weight"]
    EXPERIMENT_NAME   = cat["experiment_name"]
    MLFLOW_MODEL_NAME = cat["model_name"]
    MODEL_REGISTRY_NAME = cat["registry_name"]
    RECALL_FLOOR      = cat["recall_floor"]
    COMMON_TAGS       = {**make_common_tags(CATEGORY),
                         "run_date": pd.Timestamp.today().strftime("%Y-%m-%d")}
    mlflow.set_experiment(EXPERIMENT_NAME)
    tag = CATEGORY.lower()

    LGB_PARAMS = dict(n_estimators=600, num_leaves=63, learning_rate=0.05, subsample=0.8,
                      colsample_bytree=0.8, min_child_samples=20, reg_alpha=0.1, reg_lambda=1.0,
                      scale_pos_weight=SCALE_POS_WEIGHT, n_jobs=-1, random_state=42, verbose=-1,
                      metric='auc')  # auc early-stopping prevents trivial all-zero convergence
    with mlflow.start_run(run_name=f'lgb_{tag}') as run:
        mlflow.set_tags({**COMMON_TAGS, 'model_type': 'gradient_boosting_lgbm'})
        lgb = LGBMClassifier(**LGB_PARAMS)
        lgb.fit(X_train, y_train, eval_set=[(X_val, y_val)],
                callbacks=[early_stopping(40, verbose=False), log_evaluation(100)])
        vm = evaluate(lgb, X_val, y_val, 'val'); tm = evaluate(lgb, X_test, y_test, 'test')
        mlflow.log_params({**LGB_PARAMS, 'best_iter': lgb.best_iteration_})
        mlflow.log_metrics({**vm, **tm})
        log_cm(lgb, X_test, y_test, f'lgb_{tag}')
        plot_roc_pr(lgb, f'lgb_{tag}')
        log_feat_importance(lgb.feature_importances_, f'lgb_{tag}')
        log_clf_report(lgb, f'lgb_{tag}')
        mlflow.sklearn.log_model(lgb, 'model',
                                 signature=infer_signature(X_train, lgb.predict_proba(X_train)),
                                 input_example=X_train[:3],
                                 serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_PICKLE)
        CATEGORY_RESULTS[CATEGORY]['lgb_run'] = run.info.run_id
    print(f'  [{CATEGORY}] LGB  AUC={tm["test_auc"]}  AP={tm["test_ap"]}  F1={tm["test_f1"]}  best_iter={lgb.best_iteration_}')

In [ ]:
# ── CELL 15 : MODEL 5 — CatBoost (per category) ────────────────────────────────
for CATEGORY in ["TVM", "GATE"]:
    cat = CATEGORIES[CATEGORY]
    X_train = cat["X_train"]; y_train = cat["y_train"]
    X_val   = cat["X_val"];   y_val   = cat["y_val"]
    X_test  = cat["X_test"];  y_test  = cat["y_test"]
    X_cv    = cat["X_cv"];    y_cv    = cat["y_cv"]
    train_df = cat["train_df"]; val_df = cat["val_df"]; test_df = cat["test_df"]
    SCALE_POS_WEIGHT  = cat["scale_pos_weight"]
    EXPERIMENT_NAME   = cat["experiment_name"]
    MLFLOW_MODEL_NAME = cat["model_name"]
    MODEL_REGISTRY_NAME = cat["registry_name"]
    RECALL_FLOOR      = cat["recall_floor"]
    COMMON_TAGS       = {**make_common_tags(CATEGORY),
                         "run_date": pd.Timestamp.today().strftime("%Y-%m-%d")}
    mlflow.set_experiment(EXPERIMENT_NAME)
    tag = CATEGORY.lower()

    cat_idx = [i for i, c in enumerate(FEATURE_COLS)
               if any(k in c for k in ('device_type', 'facility', 'subsystem', 'enc'))]
    CB_PARAMS = dict(iterations=600, depth=7, learning_rate=0.05, l2_leaf_reg=3,
                     scale_pos_weight=SCALE_POS_WEIGHT, eval_metric='AUC',
                     early_stopping_rounds=40, random_seed=42, task_type='CPU', verbose=100)
    with mlflow.start_run(run_name=f'cb_{tag}') as run:
        mlflow.set_tags({**COMMON_TAGS, 'model_type': 'gradient_boosting_catboost'})
        cb = CatBoostClassifier(**CB_PARAMS)
        cb.fit(X_train, y_train, eval_set=(X_val, y_val),
               cat_features=cat_idx if cat_idx else None)
        vm = evaluate(cb, X_val, y_val, 'val'); tm = evaluate(cb, X_test, y_test, 'test')
        mlflow.log_params({**{k: v for k, v in CB_PARAMS.items() if k != 'verbose'},
                           'best_iter': cb.best_iteration_, 'cat_features_count': len(cat_idx)})
        mlflow.log_metrics({**vm, **tm})
        log_cm(cb, X_test, y_test, f'cb_{tag}')
        plot_roc_pr(cb, f'cb_{tag}')
        log_feat_importance(cb.get_feature_importance(), f'cb_{tag}')
        log_clf_report(cb, f'cb_{tag}')
        cb.save_model(f'/tmp/ps1_{tag}_catboost.cbm')
        mlflow.log_artifact(f'/tmp/ps1_{tag}_catboost.cbm', 'catboost_native')
        mlflow.sklearn.log_model(cb, 'model',
                                 signature=infer_signature(X_train, cb.predict_proba(X_train)),
                                 input_example=X_train[:3],
                                 serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_PICKLE)
        CATEGORY_RESULTS[CATEGORY]['cb_run'] = run.info.run_id
    print(f'  [{CATEGORY}] CB   AUC={tm["test_auc"]}  AP={tm["test_ap"]}  F1={tm["test_f1"]}  best_iter={cb.best_iteration_}')


In [ ]:
# ── CELL 16 : Base model leaderboard (compare 5 models per category) ───────────
for CATEGORY in ['TVM', 'GATE']:
    cat = CATEGORIES[CATEGORY]
    mlflow.set_experiment(cat['experiment_name'])
    runs = mlflow.search_runs(
        experiment_names=[cat['experiment_name']],
        filter_string="tags.model_type != 'shap_analysis' AND tags.model_type != 'xgboost_multiclass'",
        order_by=['metrics.test_ap DESC'])
    print(f'\n── {CATEGORY} base model leaderboard (ranked by test_ap) ──')
    cols = ['tags.mlflow.runName', 'metrics.test_auc', 'metrics.test_ap',
            'metrics.test_f1', 'metrics.test_prec', 'metrics.test_rec']
    available = [c for c in cols if c in runs.columns]
    print(runs[available].head(10).to_string(index=False))

In [ ]:
# ── CELL 17 : Optuna — XGBoost + LightGBM HP tuning (per category) ──────────
# XGB: 150 trials × 5-fold TimeSeriesSplit.  LGB: 80 trials × 5-fold.
for CATEGORY in ["TVM", "GATE"]:
    cat = CATEGORIES[CATEGORY]
    X_train = cat["X_train"]; y_train = cat["y_train"]
    X_val   = cat["X_val"];   y_val   = cat["y_val"]
    X_test  = cat["X_test"];  y_test  = cat["y_test"]
    X_cv    = cat["X_cv"];    y_cv    = cat["y_cv"]
    train_df = cat["train_df"]; val_df = cat["val_df"]; test_df = cat["test_df"]
    SCALE_POS_WEIGHT  = cat["scale_pos_weight"]
    EXPERIMENT_NAME   = cat["experiment_name"]
    MLFLOW_MODEL_NAME = cat["model_name"]
    MODEL_REGISTRY_NAME = cat["registry_name"]
    RECALL_FLOOR      = cat["recall_floor"]
    COMMON_TAGS       = {**make_common_tags(CATEGORY),
                         "run_date": pd.Timestamp.today().strftime("%Y-%m-%d")}
    mlflow.set_experiment(EXPERIMENT_NAME)
    tag = CATEGORY.lower()

    TS_CV = TimeSeriesSplit(n_splits=5)

    # ── XGBoost Optuna (150 trials) ──────────────────────────────────────
    def optuna_objective(trial):
        p = dict(
            n_estimators       = trial.suggest_int('n_estimators', 200, 800),
            max_depth          = trial.suggest_int('max_depth', 4, 9),
            learning_rate      = trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            subsample          = trial.suggest_float('subsample', 0.5, 1.0),
            colsample_bytree   = trial.suggest_float('colsample_bytree', 0.5, 1.0),
            min_child_weight   = trial.suggest_int('min_child_weight', 1, 30),
            gamma              = trial.suggest_float('gamma', 0.0, 3.0),
            reg_alpha          = trial.suggest_float('reg_alpha', 1e-4, 10., log=True),
            reg_lambda         = trial.suggest_float('reg_lambda', 1e-4, 10., log=True),
            scale_pos_weight   = SCALE_POS_WEIGHT,
            tree_method='hist', random_state=42, n_jobs=-1)
        scores = []
        for step, (tr, va) in enumerate(TS_CV.split(X_cv)):
            m = XGBClassifier(**p)
            m.fit(X_cv[tr], y_cv[tr], eval_set=[(X_cv[va], y_cv[va])], verbose=False)
            scores.append(roc_auc_score(y_cv[va], m.predict_proba(X_cv[va])[:, 1]))
            trial.report(np.mean(scores), step=step)
            if trial.should_prune(): raise optuna.TrialPruned()
        return np.mean(scores)

    study = optuna.create_study(direction='maximize', study_name=f'ps1-xgb-{tag}',
                                 pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1))
    study.optimize(optuna_objective, n_trials=150, show_progress_bar=True)
    BEST_PARAMS = {**study.best_params, 'scale_pos_weight': SCALE_POS_WEIGHT,
                   'tree_method': 'hist', 'random_state': 42, 'n_jobs': -1}
    CATEGORY_RESULTS[CATEGORY]['best_params'] = BEST_PARAMS

    with mlflow.start_run(run_name=f'xgb_optuna_{tag}') as run:
        mlflow.set_tags({**COMMON_TAGS, 'model_type': 'xgboost_optuna_tuned'})
        mlflow.log_params({**BEST_PARAMS, 'optuna_trials': 150, 'optuna_cv_folds': 5,
                           'optuna_best_cv': round(study.best_value, 4), 'smote': False})
        xgb_tuned = XGBClassifier(**BEST_PARAMS)
        xgb_tuned.fit(X_cv, y_cv, verbose=False)
        vm = evaluate(xgb_tuned, X_val, y_val, 'val'); tm = evaluate(xgb_tuned, X_test, y_test, 'test')
        mlflow.log_metrics({**vm, **tm, 'optuna_cv_auc': round(study.best_value, 4)})
        log_cm(xgb_tuned, X_test, y_test, f'xgb_optuna_{tag}')
        plot_roc_pr(xgb_tuned, f'xgb_optuna_{tag}')
        log_feat_importance(xgb_tuned.feature_importances_, f'xgb_optuna_{tag}')
        log_clf_report(xgb_tuned, f'xgb_optuna_{tag}')
        for fname, fig_fn in [
            (f'optuna_history_{tag}.png',      optuna.visualization.matplotlib.plot_optimization_history),
            (f'optuna_param_import_{tag}.png', optuna.visualization.matplotlib.plot_param_importances)]:
            try:
                fig_fn(study); plt.tight_layout()
                plt.savefig(f'/tmp/{fname}', dpi=80); plt.close()
                mlflow.log_artifact(f'/tmp/{fname}')
            except Exception: pass
        sig = infer_signature(pd.DataFrame(X_train, columns=FEATURE_COLS), xgb_tuned.predict_proba(X_train))
        mlflow.xgboost.log_model(xgb_tuned, 'model', signature=sig,
                                  input_example=pd.DataFrame(X_train[:3], columns=FEATURE_COLS))
        CATEGORY_RESULTS[CATEGORY]['xgb_tuned_run']   = run.info.run_id
        CATEGORY_RESULTS[CATEGORY]['xgb_tuned_model'] = xgb_tuned
    print(f'  [{CATEGORY}] XGB_Optuna  CV={study.best_value:.4f}  AUC={tm["test_auc"]}  AP={tm["test_ap"]}')

    # ── LightGBM Optuna (80 trials) ──────────────────────────────────────
    def lgb_objective(trial):
        p = dict(
            n_estimators      = trial.suggest_int('n_estimators', 200, 800),
            max_depth         = trial.suggest_int('max_depth', 4, 10),
            learning_rate     = trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            num_leaves        = trial.suggest_int('num_leaves', 20, 200),
            subsample         = trial.suggest_float('subsample', 0.5, 1.0),
            colsample_bytree  = trial.suggest_float('colsample_bytree', 0.5, 1.0),
            min_child_samples = trial.suggest_int('min_child_samples', 5, 100),
            reg_alpha         = trial.suggest_float('reg_alpha', 1e-4, 10., log=True),
            reg_lambda        = trial.suggest_float('reg_lambda', 1e-4, 10., log=True),
            scale_pos_weight  = SCALE_POS_WEIGHT,
            metric='auc', n_jobs=-1, random_state=42, verbose=-1)
        scores = []
        for step, (tr, va) in enumerate(TS_CV.split(X_cv)):
            m = LGBMClassifier(**p)
            m.fit(X_cv[tr], y_cv[tr], eval_set=[(X_cv[va], y_cv[va])])
            scores.append(roc_auc_score(y_cv[va], m.predict_proba(X_cv[va])[:, 1]))
            trial.report(np.mean(scores), step=step)
            if trial.should_prune(): raise optuna.TrialPruned()
        return np.mean(scores)

    lgb_study = optuna.create_study(
        direction='maximize', study_name=f'ps1-lgb-{tag}',
        pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1))
    lgb_study.optimize(lgb_objective, n_trials=80, show_progress_bar=True)
    BEST_LGB_PARAMS = {**lgb_study.best_params,
                       'scale_pos_weight': SCALE_POS_WEIGHT,
                       'metric': 'auc', 'n_jobs': -1, 'random_state': 42, 'verbose': -1}
    CATEGORY_RESULTS[CATEGORY]['best_lgb_params'] = BEST_LGB_PARAMS

    with mlflow.start_run(run_name=f'lgb_optuna_{tag}') as run:
        mlflow.set_tags({**COMMON_TAGS, 'model_type': 'lgbm_optuna_tuned'})
        mlflow.log_params({**BEST_LGB_PARAMS, 'optuna_trials': 80, 'optuna_cv_folds': 5,
                           'optuna_best_cv': round(lgb_study.best_value, 4)})
        lgb_tuned = LGBMClassifier(**BEST_LGB_PARAMS)
        lgb_tuned.fit(X_cv, y_cv, eval_set=[(X_val, y_val)])
        vm = evaluate(lgb_tuned, X_val, y_val, 'val'); tm = evaluate(lgb_tuned, X_test, y_test, 'test')
        mlflow.log_metrics({**vm, **tm, 'optuna_cv_auc': round(lgb_study.best_value, 4)})
        log_cm(lgb_tuned, X_test, y_test, f'lgb_optuna_{tag}')
        plot_roc_pr(lgb_tuned, f'lgb_optuna_{tag}')
        log_clf_report(lgb_tuned, f'lgb_optuna_{tag}')
        sig = infer_signature(pd.DataFrame(X_train, columns=FEATURE_COLS),
                              lgb_tuned.predict_proba(X_train))
        mlflow.sklearn.log_model(lgb_tuned, 'model', signature=sig,
                                  input_example=pd.DataFrame(X_train[:3], columns=FEATURE_COLS))
        CATEGORY_RESULTS[CATEGORY]['lgb_tuned_run']   = run.info.run_id
        CATEGORY_RESULTS[CATEGORY]['lgb_tuned_model'] = lgb_tuned
    print(f'  [{CATEGORY}] LGB_Optuna  CV={lgb_study.best_value:.4f}  AUC={tm["test_auc"]}  AP={tm["test_ap"]}')


In [ ]:
# ── CELL 18 : Stacking Ensemble (per category) ──────────────────────────────
# Base: LR + RF + XGB_tuned + LGB + CatBoost
# Final: CalibratedClassifierCV(LogisticRegression, isotonic)

# CatBoostClassifier mutates class_weights internally so sklearn clone() fails
# (get_params() doesn't return exactly what was passed to __init__).
# This thin BaseEstimator wrapper stores params as plain attrs; clone() works.
from sklearn.base import BaseEstimator, ClassifierMixin

class _CBWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, iterations=300, learning_rate=0.05,
                 class_weights=None, random_seed=42, verbose=0):
        self.iterations    = iterations
        self.learning_rate = learning_rate
        self.class_weights = class_weights
        self.random_seed   = random_seed
        self.verbose       = verbose

    def fit(self, X, y, **kw):
        self._cb = CatBoostClassifier(
            iterations=self.iterations, learning_rate=self.learning_rate,
            class_weights=self.class_weights, random_seed=self.random_seed,
            verbose=self.verbose)
        self._cb.fit(X, y, **kw)
        self.classes_ = np.array([0, 1])
        return self

    def predict(self, X):
        return self._cb.predict(X).flatten().astype(int)

    def predict_proba(self, X):
        return self._cb.predict_proba(X)


for CATEGORY in ["TVM", "GATE"]:
    cat = CATEGORIES[CATEGORY]
    X_train = cat["X_train"]; y_train = cat["y_train"]
    X_val   = cat["X_val"];   y_val   = cat["y_val"]
    X_test  = cat["X_test"];  y_test  = cat["y_test"]
    X_cv    = cat["X_cv"];    y_cv    = cat["y_cv"]
    train_df = cat["train_df"]; val_df = cat["val_df"]; test_df = cat["test_df"]
    SCALE_POS_WEIGHT  = cat["scale_pos_weight"]
    EXPERIMENT_NAME   = cat["experiment_name"]
    MLFLOW_MODEL_NAME = cat["model_name"]
    MODEL_REGISTRY_NAME = cat["registry_name"]
    RECALL_FLOOR      = cat["recall_floor"]
    COMMON_TAGS       = {**make_common_tags(CATEGORY),
                         "run_date": pd.Timestamp.today().strftime("%Y-%m-%d")}
    mlflow.set_experiment(EXPERIMENT_NAME)
    tag = CATEGORY.lower()

    BEST_PARAMS = CATEGORY_RESULTS[CATEGORY]['best_params']

    with mlflow.start_run(run_name=f'stack_{tag}') as run:
        mlflow.set_tags({**COMMON_TAGS, 'model_type': 'stacking_ensemble'})
        mlflow.log_params({'base_models': 'LR+RF+XGB_tuned+LGB+CatBoost',
                           'final_estimator': 'LogisticRegression+isotonic',
                           'cv_strategy': 'TimeSeriesSplit(5)', 'passthrough': True})
        base = [
            ('lr',  Pipeline([('sc', StandardScaler()),
                              ('clf', LogisticRegression(C=0.1, class_weight='balanced',
                                                         max_iter=1000, random_state=42))])),
            ('rf',  RandomForestClassifier(n_estimators=200, max_depth=10,
                                           class_weight='balanced_subsample', n_jobs=-1, random_state=42)),
            ('xgb', XGBClassifier(**{**BEST_PARAMS, 'n_estimators': 300}, verbosity=0)),
            ('lgb', LGBMClassifier(n_estimators=300, learning_rate=0.05,
                                   scale_pos_weight=SCALE_POS_WEIGHT, n_jobs=-1, random_state=42, verbose=-1)),
            ('cb',  _CBWrapper(iterations=300, learning_rate=0.05,
                               class_weights=[1, SCALE_POS_WEIGHT], random_seed=42, verbose=0)),
        ]
        stack = StackingClassifier(
            estimators=base,
            final_estimator=CalibratedClassifierCV(
                LogisticRegression(C=1.0, max_iter=500), cv=3, method='isotonic'),
            cv=TimeSeriesSplit(n_splits=5), stack_method='predict_proba',
            passthrough=True, n_jobs=-1)
        stack.fit(X_cv, y_cv)
        vm = evaluate(stack, X_val, y_val, 'val'); tm = evaluate(stack, X_test, y_test, 'test')
        mlflow.log_metrics({**vm, **tm})
        log_cm(stack, X_test, y_test, f'stack_{tag}')
        plot_roc_pr(stack, f'stack_{tag}')
        log_clf_report(stack, f'stack_{tag}')
        mlflow.sklearn.log_model(stack, 'model',
                                 signature=infer_signature(X_train, stack.predict_proba(X_train)),
                                 input_example=X_train[:3],
                                 serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_PICKLE)
        CATEGORY_RESULTS[CATEGORY]['stack_run'] = run.info.run_id
    print(f'  [{CATEGORY}] Stack  AUC={tm["test_auc"]}  AP={tm["test_ap"]}  F1={tm["test_f1"]}')


In [ ]:
# ── CELL 19 : SHAP — feature attribution (per category, on Optuna XGB) ─────────
for CATEGORY in ["TVM", "GATE"]:
    cat = CATEGORIES[CATEGORY]
    X_train = cat["X_train"]; y_train = cat["y_train"]
    X_val   = cat["X_val"];   y_val   = cat["y_val"]
    X_test  = cat["X_test"];  y_test  = cat["y_test"]
    X_cv    = cat["X_cv"];    y_cv    = cat["y_cv"]
    train_df = cat["train_df"]; val_df = cat["val_df"]; test_df = cat["test_df"]
    SCALE_POS_WEIGHT  = cat["scale_pos_weight"]
    EXPERIMENT_NAME   = cat["experiment_name"]
    MLFLOW_MODEL_NAME = cat["model_name"]
    MODEL_REGISTRY_NAME = cat["registry_name"]
    RECALL_FLOOR      = cat["recall_floor"]
    COMMON_TAGS       = {**make_common_tags(CATEGORY),
                         "run_date": pd.Timestamp.today().strftime("%Y-%m-%d")}
    mlflow.set_experiment(EXPERIMENT_NAME)
    tag = CATEGORY.lower()

    xgb_tuned = CATEGORY_RESULTS[CATEGORY]['xgb_tuned_model']
    X_shap = pd.DataFrame(X_test[:2000], columns=FEATURE_COLS)
    explainer = shap.TreeExplainer(xgb_tuned)
    shap_values = explainer(X_shap)

    with mlflow.start_run(run_name=f'xgb_shap_{tag}') as run:
        mlflow.set_tags({**COMMON_TAGS, 'model_type': 'shap_analysis',
                         'parent_run': CATEGORY_RESULTS[CATEGORY]['xgb_tuned_run']})
        for sname, sfn in [('bar', shap.plots.bar), ('beeswarm', shap.plots.beeswarm)]:
            sfn(shap_values, max_display=20, show=False)
            fig = plt.gcf(); fig.tight_layout()
            fpath = f'/tmp/shap_{sname}_{tag}.png'
            fig.savefig(fpath, dpi=100); plt.close(fig); mlflow.log_artifact(fpath)
        top_feat = pd.Series(np.abs(shap_values.values).mean(0), index=FEATURE_COLS).idxmax()
        shap.plots.scatter(shap_values[:, top_feat], color=shap_values, show=False)
        fig = plt.gcf(); fig.tight_layout()
        fpath = f'/tmp/shap_dep_{tag}.png'
        fig.savefig(fpath, dpi=100); plt.close(fig); mlflow.log_artifact(fpath)
        shap_imp = (pd.Series(np.abs(shap_values.values).mean(0), index=FEATURE_COLS)
                      .sort_values(ascending=False).reset_index()
                      .rename(columns={'index': 'feature', 0: 'mean_abs_shap'}))
        shap_imp.to_csv(f'/tmp/shap_importance_{tag}.csv', index=False)
        mlflow.log_artifact(f'/tmp/shap_importance_{tag}.csv')
        mlflow.log_param('top_shap_feature', top_feat)
    print(f'  [{CATEGORY}] Top SHAP feature: {top_feat}')


In [ ]:
# ── CELL 20 : Stage 2 — severity multiclass (per category, guarded) ────────────
# Skipped for GATE: failure_levels 2/3/4 have ~6/7/6 rows — min class < 20.
for CATEGORY in ["TVM", "GATE"]:
    cat = CATEGORIES[CATEGORY]
    X_train = cat["X_train"]; y_train = cat["y_train"]
    X_val   = cat["X_val"];   y_val   = cat["y_val"]
    X_test  = cat["X_test"];  y_test  = cat["y_test"]
    X_cv    = cat["X_cv"];    y_cv    = cat["y_cv"]
    train_df = cat["train_df"]; val_df = cat["val_df"]; test_df = cat["test_df"]
    SCALE_POS_WEIGHT  = cat["scale_pos_weight"]
    EXPERIMENT_NAME   = cat["experiment_name"]
    MLFLOW_MODEL_NAME = cat["model_name"]
    MODEL_REGISTRY_NAME = cat["registry_name"]
    RECALL_FLOOR      = cat["recall_floor"]
    COMMON_TAGS       = {**make_common_tags(CATEGORY),
                         "run_date": pd.Timestamp.today().strftime("%Y-%m-%d")}
    mlflow.set_experiment(EXPERIMENT_NAME)
    tag = CATEGORY.lower()

    if 'failure_level' not in train_df.columns:
        print(f'  [{CATEGORY}] failure_level column absent — skip Stage 2'); continue

    sev_tr = train_df[train_df['will_fail_3d'] == 1].dropna(subset=['failure_level'])
    sev_te = test_df[test_df['will_fail_3d'] == 1].dropna(subset=['failure_level'])
    _class_counts = sev_tr['failure_level'].value_counts() if len(sev_tr) > 0 else pd.Series([])
    _skip = len(sev_tr) == 0 or _class_counts.min() < 20
    if _skip:
        print(f'  [{CATEGORY}] Stage 2 skipped — min class count'
              f" {_class_counts.min() if len(_class_counts) else 0} < 20")
        continue

    Xs_tr = sev_tr[FEATURE_COLS].values; ys_tr = sev_tr['failure_level'].values.astype(int) - 1
    Xs_te = sev_te[FEATURE_COLS].values; ys_te = sev_te['failure_level'].values.astype(int) - 1
    with mlflow.start_run(run_name=f'stage2_severity_{tag}') as run:
        mlflow.set_tags({**COMMON_TAGS, 'model_type': 'xgboost_multiclass',
                         'stage': '2', 'num_classes': '3'})
        sev_model = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,
                                   objective='multi:softprob', num_class=3,
                                   tree_method='hist', random_state=42, n_jobs=-1)
        sev_model.fit(Xs_tr, ys_tr, verbose=False)
        sev_auc = roc_auc_score(ys_te, sev_model.predict_proba(Xs_te),
                                multi_class='ovr', average='macro')
        sev_f1 = f1_score(ys_te, sev_model.predict(Xs_te), average='macro')
        mlflow.log_metrics({'test_severity_auc': round(sev_auc, 4),
                            'test_severity_f1': round(sev_f1, 4)})
        CATEGORY_RESULTS[CATEGORY]['severity_model'] = sev_model
    print(f'  [{CATEGORY}] Stage2  AUC={sev_auc:.4f}  F1={sev_f1:.4f}')


In [ ]:
# ── CELL 21 : Threshold tuning + Champion selection (per category) ──────────────
# Threshold: tune on val PR curve, target recall >= recall_floor.
# Champion: ranked by test_ap (Average Precision) — right metric under heavy imbalance.
for CATEGORY in ["TVM", "GATE"]:
    cat = CATEGORIES[CATEGORY]
    X_train = cat["X_train"]; y_train = cat["y_train"]
    X_val   = cat["X_val"];   y_val   = cat["y_val"]
    X_test  = cat["X_test"];  y_test  = cat["y_test"]
    X_cv    = cat["X_cv"];    y_cv    = cat["y_cv"]
    train_df = cat["train_df"]; val_df = cat["val_df"]; test_df = cat["test_df"]
    SCALE_POS_WEIGHT  = cat["scale_pos_weight"]
    EXPERIMENT_NAME   = cat["experiment_name"]
    MLFLOW_MODEL_NAME = cat["model_name"]
    MODEL_REGISTRY_NAME = cat["registry_name"]
    RECALL_FLOOR      = cat["recall_floor"]
    COMMON_TAGS       = {**make_common_tags(CATEGORY),
                         "run_date": pd.Timestamp.today().strftime("%Y-%m-%d")}
    mlflow.set_experiment(EXPERIMENT_NAME)
    tag = CATEGORY.lower()

    # ── Threshold tuning ─────────────────────────────────────────────────────────
    xgb_tuned = CATEGORY_RESULTS[CATEGORY]['xgb_tuned_model']
    y_prob_val_thresh = xgb_tuned.predict_proba(X_val)[:, 1]
    prec_curve, rec_curve, thresh_curve = precision_recall_curve(y_val, y_prob_val_thresh)
    candidates = [(t, p, r) for p, r, t in zip(prec_curve[:-1], rec_curve[:-1], thresh_curve)
                  if r >= RECALL_FLOOR]
    if candidates:
        BEST_THRESHOLD, BEST_PREC, BEST_REC = max(candidates, key=lambda x: x[1])
    else:
        BEST_THRESHOLD, BEST_PREC, BEST_REC = 0.5, float('nan'), float('nan')
        print(f'  [{CATEGORY}] No threshold achieves recall>={RECALL_FLOOR} — using 0.5')
    print(f'  [{CATEGORY}] Threshold: {BEST_THRESHOLD:.3f}'
          f'  prec={BEST_PREC:.3f}  rec={BEST_REC:.3f}  (recall>={RECALL_FLOOR} on val)')

    # ── Champion selection ────────────────────────────────────────────────────────
    all_runs = mlflow.search_runs(
        experiment_names=[EXPERIMENT_NAME],
        filter_string="tags.model_type != 'shap_analysis' AND tags.model_type != 'xgboost_multiclass'",
        order_by=['metrics.test_ap DESC'])
    CHAMPION_RUN_ID = all_runs.iloc[0]['run_id']
    CHAMPION_NAME   = all_runs.iloc[0]['tags.mlflow.runName']
    CHAMPION_AUC    = all_runs.iloc[0]['metrics.test_auc']
    CHAMPION_AP     = all_runs.iloc[0]['metrics.test_ap']
    print(f'  [{CATEGORY}] Champion: {CHAMPION_NAME}  AUC={CHAMPION_AUC:.4f}  AP={CHAMPION_AP:.4f}')

    CATEGORY_RESULTS[CATEGORY].update({
        'champion_run_id': CHAMPION_RUN_ID, 'champion_name': CHAMPION_NAME,
        'champion_auc': CHAMPION_AUC, 'champion_ap': CHAMPION_AP,
        'threshold': BEST_THRESHOLD, 'threshold_prec': BEST_PREC, 'threshold_rec': BEST_REC,
    })

In [ ]:
# ── CELL 22 : Register champion in MLflow Model Registry (per category) ─────────
for CATEGORY in ["TVM", "GATE"]:
    cat = CATEGORIES[CATEGORY]
    X_train = cat["X_train"]; y_train = cat["y_train"]
    X_val   = cat["X_val"];   y_val   = cat["y_val"]
    X_test  = cat["X_test"];  y_test  = cat["y_test"]
    X_cv    = cat["X_cv"];    y_cv    = cat["y_cv"]
    train_df = cat["train_df"]; val_df = cat["val_df"]; test_df = cat["test_df"]
    SCALE_POS_WEIGHT  = cat["scale_pos_weight"]
    EXPERIMENT_NAME   = cat["experiment_name"]
    MLFLOW_MODEL_NAME = cat["model_name"]
    MODEL_REGISTRY_NAME = cat["registry_name"]
    RECALL_FLOOR      = cat["recall_floor"]
    COMMON_TAGS       = {**make_common_tags(CATEGORY),
                         "run_date": pd.Timestamp.today().strftime("%Y-%m-%d")}
    mlflow.set_experiment(EXPERIMENT_NAME)
    tag = CATEGORY.lower()

    CHAMPION_RUN_ID = CATEGORY_RESULTS[CATEGORY]['champion_run_id']
    CHAMPION_NAME   = CATEGORY_RESULTS[CATEGORY]['champion_name']
    CHAMPION_AP     = CATEGORY_RESULTS[CATEGORY]['champion_ap']
    CHAMPION_AUC    = CATEGORY_RESULTS[CATEGORY]['champion_auc']

    mlfc = mlflow.tracking.MlflowClient()
    try:
        mlfc.create_registered_model(
            name=MLFLOW_MODEL_NAME,
            description=f'PS1-3d {CATEGORY} Device Failure — Chicago Ventra. Binary: will_fail_3d. v6.',
            tags={'ps': 'PS1_3d', 'category': CATEGORY, 'target': 'will_fail_3d', 'city': 'chicago'})
    except mlflow.exceptions.MlflowException: pass
    mv = mlfc.create_model_version(
        name=MLFLOW_MODEL_NAME, source=f'runs:/{CHAMPION_RUN_ID}/model',
        run_id=CHAMPION_RUN_ID,
        description=f'Champion: {CHAMPION_NAME}  AUC={CHAMPION_AUC:.4f}  AP={CHAMPION_AP:.4f}',
        tags={'algorithm': CHAMPION_NAME, 'test_ap': str(round(CHAMPION_AP, 4)),
              'test_auc': str(round(CHAMPION_AUC, 4)), 'n_features': str(len(FEATURE_COLS))})
    mlfc.transition_model_version_stage(name=MLFLOW_MODEL_NAME, version=mv.version, stage='Staging')
    CATEGORY_RESULTS[CATEGORY]['mlflow_version'] = mv.version
    print(f'  [{CATEGORY}] MLflow: {MLFLOW_MODEL_NAME} v{mv.version} → Staging')


In [ ]:
# ── CELL 23 : Register champion in SageMaker Model Registry (per category) ──────
for CATEGORY in ["TVM", "GATE"]:
    cat = CATEGORIES[CATEGORY]
    X_train = cat["X_train"]; y_train = cat["y_train"]
    X_val   = cat["X_val"];   y_val   = cat["y_val"]
    X_test  = cat["X_test"];  y_test  = cat["y_test"]
    X_cv    = cat["X_cv"];    y_cv    = cat["y_cv"]
    train_df = cat["train_df"]; val_df = cat["val_df"]; test_df = cat["test_df"]
    SCALE_POS_WEIGHT  = cat["scale_pos_weight"]
    EXPERIMENT_NAME   = cat["experiment_name"]
    MLFLOW_MODEL_NAME = cat["model_name"]
    MODEL_REGISTRY_NAME = cat["registry_name"]
    RECALL_FLOOR      = cat["recall_floor"]
    COMMON_TAGS       = {**make_common_tags(CATEGORY),
                         "run_date": pd.Timestamp.today().strftime("%Y-%m-%d")}
    mlflow.set_experiment(EXPERIMENT_NAME)
    tag = CATEGORY.lower()

    import tarfile, sagemaker.image_uris as sm_images
    CHAMPION_RUN_ID  = CATEGORY_RESULTS[CATEGORY]['champion_run_id']
    CHAMPION_NAME    = CATEGORY_RESULTS[CATEGORY]['champion_name']
    CHAMPION_AP      = CATEGORY_RESULTS[CATEGORY]['champion_ap']
    CHAMPION_AUC     = CATEGORY_RESULTS[CATEGORY]['champion_auc']
    MLFLOW_VERSION   = CATEGORY_RESULTS[CATEGORY]['mlflow_version']

    champion_model = mlflow.sklearn.load_model(f'runs:/{CHAMPION_RUN_ID}/model')
    joblib.dump(champion_model, f'/tmp/ps1_{tag}_champion.joblib')
    joblib.dump(FEATURE_COLS,   f'/tmp/ps1_{tag}_feature_cols.joblib')
    test_metrics = evaluate(champion_model, cat['X_test'], cat['y_test'], 'test')
    S3_CLIENT.put_object(Bucket=ARTIFACT_BUCKET,
                          Key=f'sagemaker/ps1-3d/{tag}/metrics/test_metrics.json',
                          Body=json.dumps(test_metrics), ContentType='application/json')
    with tarfile.open(f'/tmp/model_{tag}.tar.gz', 'w:gz') as tar:
        tar.add(f'/tmp/ps1_{tag}_champion.joblib',    arcname=f'ps1_{tag}_champion.joblib')
        tar.add(f'/tmp/ps1_{tag}_feature_cols.joblib', arcname=f'ps1_{tag}_feature_cols.joblib')
    MODEL_S3_KEY = f'sagemaker/ps1-3d/{tag}/model-v{MLFLOW_VERSION}/model.tar.gz'
    S3_CLIENT.upload_file(f'/tmp/model_{tag}.tar.gz', ARTIFACT_BUCKET, MODEL_S3_KEY)
    MODEL_DATA_URL = f's3://{ARTIFACT_BUCKET}/{MODEL_S3_KEY}'

    image_uri = sm_images.retrieve(framework='sklearn', region=REGION, version='1.2-1',
                                    instance_type='ml.m5.large', image_scope='inference')
    try:
        SM_CLIENT.create_model_package_group(
            ModelPackageGroupName=MODEL_REGISTRY_NAME,
            ModelPackageGroupDescription=f'PS1-3d {CATEGORY} — Chicago Ventra. will_fail_3d. v6.')
    except Exception: pass
    resp = SM_CLIENT.create_model_package(
        ModelPackageGroupName=MODEL_REGISTRY_NAME,
        ModelPackageDescription=f'{CHAMPION_NAME} | AP={CHAMPION_AP:.4f} | MLflow v{MLFLOW_VERSION}',
        InferenceSpecification={
            'Containers': [{'Image': image_uri, 'ModelDataUrl': MODEL_DATA_URL}],
            'SupportedContentTypes': ['text/csv', 'application/json'],
            'SupportedResponseMIMETypes': ['application/json'],
            'SupportedTransformInstanceTypes': ['ml.m5.large'],
            'SupportedRealtimeInferenceInstanceTypes': ['ml.m5.large'],
        },
        ModelApprovalStatus='PendingManualApproval',
        ModelMetrics={'ModelQuality': {'Statistics': {'ContentType': 'application/json',
            'S3Uri': f's3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/{tag}/metrics/test_metrics.json'}}},
        CustomerMetadataProperties={
            'ps': 'PS1_3d', 'category': CATEGORY, 'target': 'will_fail_3d', 'city': 'chicago',
            'algorithm': CHAMPION_NAME, 'test_ap': str(round(CHAMPION_AP, 4)),
            'test_auc': str(round(CHAMPION_AUC, 4)),
            'mlflow_version': str(MLFLOW_VERSION), 'mlflow_run_id': CHAMPION_RUN_ID,
            'n_features': str(len(FEATURE_COLS))})
    SM_MODEL_PKG_ARN = resp['ModelPackageArn']
    CATEGORY_RESULTS[CATEGORY].update({
        'sm_pkg_arn': SM_MODEL_PKG_ARN, 'model_data_url': MODEL_DATA_URL,
        'champion_model': champion_model, 'X_test': cat['X_test'], 'y_test': cat['y_test'],
        'X_train': cat['X_train'],
    })
    print(f'  [{CATEGORY}] SM Package: ...{SM_MODEL_PKG_ARN[-40:]}')

print('\nAll categories registered in MLflow + SageMaker.')
for cn, res in CATEGORY_RESULTS.items():
    print(f"  {cn:<6}  champion={res['champion_name']:<30}  ",
          f"AUC={res['champion_auc']:.4f}  AP={res['champion_ap']:.4f}  thresh={res['threshold']:.3f}")


In [ ]:
# ── CELL 11 : Approve + deploy SageMaker Endpoints (per category) ────────────
from sagemaker.model import ModelPackage
from sagemaker.model_monitor import DataCaptureConfig

PREDICTORS = {}
for CATEGORY, res in CATEGORY_RESULTS.items():
    cat = CATEGORIES[CATEGORY]
    SM_CLIENT.update_model_package(ModelPackageArn=res["sm_pkg_arn"], ModelApprovalStatus="Approved")
    print(f"[{CATEGORY}] Approved.")
    registered = ModelPackage(role=ROLE, model_package_arn=res["sm_pkg_arn"], sagemaker_session=SM_SESSION)
    predictor = registered.deploy(
        initial_instance_count=1, instance_type="ml.m5.large",
        endpoint_name=cat["endpoint"],
        data_capture_config=DataCaptureConfig(
            enable_capture=True, sampling_percentage=100,
            destination_s3_uri=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/{CATEGORY.lower()}/data-capture/",
            capture_options=["Input","Output"]),
    )
    PREDICTORS[CATEGORY] = predictor
    print(f"[{CATEGORY}] Endpoint live: {cat['endpoint']}")
    sample_csv = pd.DataFrame(res["X_test"][:3], columns=FEATURE_COLS).to_csv(index=False, header=False)
    result = predictor.predict(sample_csv, initial_args={"ContentType":"text/csv"})
    print(f"[{CATEGORY}] Sample predictions: {result}")


In [ ]:
# ── CELL 12 : SageMaker Model Monitor — Data Quality Baselines (per category) ─
from sagemaker.model_monitor import DefaultModelMonitor
from sagemaker.model_monitor.dataset_format import DatasetFormat
from sagemaker.model_monitor import CronExpressionGenerator

MONITORS = {}
for CATEGORY, res in CATEGORY_RESULTS.items():
    tag = CATEGORY.lower()
    monitor = DefaultModelMonitor(role=ROLE, instance_count=1, instance_type="ml.m5.large",
                                   volume_size_in_gb=20, max_runtime_in_seconds=3600,
                                   sagemaker_session=SM_SESSION)
    baseline_df = pd.DataFrame(res["X_train"], columns=FEATURE_COLS)
    baseline_df.to_csv(f"/tmp/baseline_{tag}.csv", index=False)
    S3_CLIENT.upload_file(f"/tmp/baseline_{tag}.csv", ARTIFACT_BUCKET,
                          f"sagemaker/ps1-3d/{tag}/baseline/baseline.csv")
    monitor.suggest_baseline(
        baseline_dataset=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/{tag}/baseline/baseline.csv",
        dataset_format=DatasetFormat.csv(header=True),
        output_s3_uri=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/{tag}/baseline/output/",
        wait=True, logs=True)
    print(f"[{CATEGORY}] Baseline complete.")
    MONITORS[CATEGORY] = monitor
print("All baselines done.")


In [ ]:
# ── CELL 13 : Schedule daily Data Quality monitoring (per category) ───────────
cw = boto3.client("cloudwatch", region_name=REGION)

for CATEGORY, monitor in MONITORS.items():
    tag = CATEGORY.lower()
    cat = CATEGORIES[CATEGORY]
    sched_name = f"chicago-ps1-3d-{tag}-data-quality"
    monitor.create_monitoring_schedule(
        monitor_schedule_name=sched_name, endpoint_input=cat["endpoint"],
        output_s3_uri=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/{tag}/monitor-output/",
        statistics=monitor.baseline_statistics(), constraints=monitor.suggested_constraints(),
        schedule_cron_expression=CronExpressionGenerator.daily(), enable_cloudwatch_metrics=True)
    print(f"[{CATEGORY}] Data Quality Monitor: {sched_name}")
    cw.put_metric_alarm(
        AlarmName=f"PS1-3d-{CATEGORY}-DataQuality-ViolationRate",
        ComparisonOperator="GreaterThanThreshold", EvaluationPeriods=1,
        MetricName="feature_baseline_drift_percentage",
        Namespace="aws/sagemaker/Endpoints/data-metrics",
        Period=86400, Statistic="Average", Threshold=3.0,
        AlarmDescription=f"PS1-3d {CATEGORY} input feature violation rate > 3%",
        Dimensions=[{"Name":"Endpoint","Value":cat["endpoint"]}],
        TreatMissingData="notBreaching")
    print(f"[{CATEGORY}] CW Alarm: PS1-3d-{CATEGORY}-DataQuality-ViolationRate")


In [ ]:
# ── CELL 14 : Model Quality Monitor (per category) ──────────────────────────
from sagemaker.model_monitor import ModelQualityMonitor
from sagemaker.model_monitor.dataset_format import DatasetFormat

MQ_MONITORS = {}
for CATEGORY, res in CATEGORY_RESULTS.items():
    tag = CATEGORY.lower()
    cat = CATEGORIES[CATEGORY]
    mq = ModelQualityMonitor(role=ROLE, instance_count=1, instance_type="ml.m5.large",
                              sagemaker_session=SM_SESSION)
    y_prob_test = res["champion_model"].predict_proba(res["X_test"])[:,1]
    mq_df = pd.DataFrame({"prediction":y_prob_test, "probability":y_prob_test,
                           "label":res["y_test"].astype(str)})
    mq_df.to_csv(f"/tmp/mq_baseline_{tag}.csv", index=False)
    S3_CLIENT.upload_file(f"/tmp/mq_baseline_{tag}.csv", ARTIFACT_BUCKET,
                          f"sagemaker/ps1-3d/{tag}/mq-baseline/mq_baseline.csv")
    mq.suggest_baseline(
        baseline_dataset=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/{tag}/mq-baseline/mq_baseline.csv",
        dataset_format=DatasetFormat.csv(header=True),
        output_s3_uri=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/{tag}/mq-baseline/output/",
        problem_type="BinaryClassification",
        inference_attribute="prediction", probability_attribute="probability",
        ground_truth_attribute="label", probability_threshold_attribute="0.5", wait=True)
    mq.create_monitoring_schedule(
        monitor_schedule_name=f"chicago-ps1-3d-{tag}-model-quality",
        endpoint_input=cat["endpoint"],
        ground_truth_input=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/{tag}/ground-truth/",
        problem_type="BinaryClassification",
        output_s3_uri=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/{tag}/mq-output/",
        constraints=mq.suggested_constraints(),
        schedule_cron_expression=CronExpressionGenerator.daily(), enable_cloudwatch_metrics=True)
    print(f"[{CATEGORY}] Model Quality Monitor: chicago-ps1-3d-{tag}-model-quality")
    print(f"  Ground truth: s3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/{tag}/ground-truth/")
    MQ_MONITORS[CATEGORY] = mq


In [ ]:
# ── CELL 15 : Final summary (per category) ────────────────────────────────────
print("="*65)
print("PS1-3d TVM+GATE -- Complete  (CUBIC MARS AWS v6)")
print("="*65)

for CATEGORY, res in CATEGORY_RESULTS.items():
    cat = CATEGORIES[CATEGORY]
    tag = CATEGORY.lower()
    print(f"\n-- {CATEGORY} --")
    print(f"  Champion       : {res['champion_name']}  AUC={res['champion_auc']:.4f}")
    print(f"  MLflow model   : {cat['model_name']}  v{res['mlflow_version']}  [Staging]")
    print(f"  SM Registry    : {cat['registry_name']}")
    print(f"  SM Endpoint    : {cat['endpoint']}")
    print(f"  Feature Store  : {cat['fs_group']}")
    print(f"  Data Monitor   : chicago-ps1-3d-{tag}-data-quality  (daily)")
    print(f"  Model Monitor  : chicago-ps1-3d-{tag}-model-quality  (daily)")
    print(f"  CW Alarm       : PS1-3d-{CATEGORY}-DataQuality-ViolationRate")

print(f"\n  Features       : {len(FEATURE_COLS)}")
print(f"  Artifacts      : s3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/")
print("""
Retraining flow:
  1. Re-run from CELL 4 -- reads latest Gold, trains both TVM + GATE
  2. MLflow auto-increments version numbers per category
  3. SM Registry adds new packages (PendingManualApproval)
  4. Approve in Studio UI -> new Endpoints deployed
""")
